# Lab 1 baseline validation (Gemini stub)

This notebook mirrors `python -m src.app` but patches the Gemini client so we can exercise schema validation and report generation offline.

In [ ]:
from types import SimpleNamespace
from unittest.mock import patch, MagicMock
import json
from pathlib import Path
from src import app

In [ ]:
def run_stubbed_pipeline():
    sample_findings = [
        {
            'title': 'Mocked OWASP risk',
            'severity': 'medium',
            'rationale': 'Stubbed response for testing without external API',
            'cwe': 'CWE-20',
        }
    ]

    def fake_generate_content(*args, **kwargs):
        payload = {
            'llm_risks': ['LLM01'],
            'findings': sample_findings,
        }
        return SimpleNamespace(text=json.dumps(payload))

    with patch('src.app.genai.Client') as MockClient:
        instance = MagicMock()
        instance.models.generate_content.side_effect = fake_generate_content
        MockClient.return_value = instance
        app.main(client=instance, model_id="offline-fixture")

    report_path = Path('reports/baseline.json')
    print('Report written to:', report_path.resolve())
    return json.loads(report_path.read_text())

results = run_stubbed_pipeline()
results[0]